In [1]:
from cropspectai import ml_config
from cropspectai.modeling.predict import predict_model
from cropspectai.data_analysis import feature_engineering
import pandas as pd


# --------------------------------------------------
# Load data
# --------------------------------------------------

#basil data
# X_new = pd.read_csv(ml_config.BASIL_EXPERIMENT_DATA_PATH / "X_new.csv", index_col = False)

#misc data
X_new = pd.read_csv(ml_config.PREDICTION_DATA_DIR / "misc" / "X_new.csv", index_col = False)

print("Shape of imported data: ", X_new.shape)

Shape of imported data:  (20, 15)


In [2]:
column_list = X_new.columns.to_list()
print(column_list)

['timestamp', 'specimen', 'noisy_channels', 'F1', 'F2', 'FZ', 'F3', 'F4', 'FY', 'F5', 'FXL', 'F6', 'F7', 'F8', 'NIR']


In [3]:
AS7343_CHANNELS = list(ml_config.AS7343_CHANNELS.keys())
print(AS7343_CHANNELS)

['F1', 'F2', 'FZ', 'F3', 'F4', 'FY', 'F5', 'FXL', 'F6', 'F7', 'F8', 'NIR']


In [4]:
non_features = column_list[:3]
print(non_features)
X_non_features = X_new[non_features].copy()
print(X_non_features.shape)

['timestamp', 'specimen', 'noisy_channels']
(20, 3)


In [5]:
X_new = X_new[AS7343_CHANNELS].copy()


In [6]:
X_new = feature_engineering.add_all_features(X_new).copy()
print(X_new.shape)

(20, 40)


In [7]:
X_new.columns

Index(['F1', 'F2', 'FZ', 'F3', 'F4', 'FY', 'F5', 'FXL', 'F6', 'F7', 'F8',
       'NIR', 'ci_red_edge_as7343', 'ci_green_as7343', 'gm_nd_as7343',
       'red_edge_ratio_as7343', 'datt2_as7343', 'datt1_as7343', 'mcari_as7343',
       'tcari_as7343', 'osavi_as7343', 'mcari_osavi_as7343',
       'tcari_osavi_as7343', 'mndvi_as7343', 'ndvi_as7343', 'gndvi_as7343',
       'sr_as7343', 'green_sr_as7343', 'evi2_as7343', 'rdvi_as7343',
       'sipi_as7343', 'psri_as7343', 'red_green_nd', 'red_edge_green_nd',
       'nir_green_nd', 'red_edge_slope_proxy', 'green_red_ratio',
       'rededge_green_ratio', 'nir_fy_ratio', 'nir_fxl_ratio'],
      dtype='str')

## import model config json file for feature list

In [8]:
import json

model_config_path = ml_config.MODELS_DIR / "xgb_best_parameters_v_01.json"
with open(model_config_path, "r") as f:
    model_config = json.load(f)
features = model_config["features"]
print(features)

['F1', 'F2', 'FZ', 'F3', 'F4', 'FY', 'F5', 'FXL', 'F6', 'F7', 'F8', 'NIR', 'mndvi_as7343', 'sipi_as7343', 'ci_red_edge_as7343', 'psri_as7343', 'red_green_nd', 'datt1_as7343', 'mcari_osavi_as7343']


### remove columns not in feature list

In [9]:
X_new = X_new[features].copy()
X_new.columns

Index(['F1', 'F2', 'FZ', 'F3', 'F4', 'FY', 'F5', 'FXL', 'F6', 'F7', 'F8',
       'NIR', 'mndvi_as7343', 'sipi_as7343', 'ci_red_edge_as7343',
       'psri_as7343', 'red_green_nd', 'datt1_as7343', 'mcari_osavi_as7343'],
      dtype='str')

In [10]:
X_new.tail(20)

,F1,F2,FZ,F3,F4,FY,F5,FXL,F6,F7,F8,NIR,mndvi_as7343,sipi_as7343,ci_red_edge_as7343,psri_as7343,red_green_nd,datt1_as7343,mcari_osavi_as7343
0,0.00,0.000000,0.000000,0.003788,0.055586,0.068423,0.104054,0.052744,0.018185,0.072422,0.333333,0.125000,0.266323,2.377420,-0.625000,0.050508,-0.179243,-3.962361,5.800654
1,0.09,0.018868,0.011811,0.011364,0.097276,0.114608,0.164865,0.093786,0.041494,0.104287,0.410256,0.166667,0.252212,2.482462,-0.593749,0.017089,-0.225070,-3.904921,6.018675
2,0.08,0.018868,0.011811,0.011364,0.097276,0.116319,0.168919,0.093951,0.042652,0.104867,0.410256,0.166667,0.249282,2.505761,-0.593749,0.018503,-0.233949,-3.941570,6.055126
3,0.01,0.018868,0.007874,0.011364,0.088938,0.106911,0.155405,0.085710,0.038023,0.098494,0.384615,0.166667,0.273334,2.329265,-0.566665,0.024846,-0.224148,-3.196984,5.044467
4,0.05,0.018868,0.007874,0.007955,0.097276,0.112898,0.168919,0.089171,0.038023,0.107184,0.461538,0.166667,0.230462,2.669553,-0.638888,0.021467,-0.223594,-4.957231,8.009580
5,0.00,0.015094,0.003937,0.006439,0.080600,0.099213,0.141892,0.084061,0.042982,0.098494,0.358974,0.166667,0.264969,2.387015,-0.535713,0.049848,-0.180535,-2.820867,4.253273
6,0.00,0.011321,0.003937,0.007576,0.075042,0.094081,0.128378,0.079117,0.040007,0.089803,0.333333,0.125000,0.170092,3.439583,-0.625000,0.044283,-0.176803,-5.919056,6.901374
7,0.00,0.000000,0.000000,0.000000,0.058366,0.070133,0.114865,0.051096,0.013225,0.086906,0.461538,0.166667,0.314548,2.089580,-0.638888,0.061837,-0.138568,-3.696932,7.247436
8,0.00,0.000000,0.000000,0.000000,0.058366,0.069962,0.114865,0.051096,0.013225,0.086906,0.461538,0.166667,0.314548,2.089580,-0.638888,0.061837,-0.138568,-3.696932,7.247436
9,0.00,0.000000,0.000000,0.000000,0.058366,0.070133,0.114865,0.051096,0.013225,0.086906,0.461538,0.166667,0.314548,2.089580,-0.638888,0.061837,-0.138568,-3.696932,7.247436


## load best xgb model

In [11]:
model_path = ml_config.MODELS_DIR / "xgb_v_01.joblib"

In [12]:
predictions = predict_model(X_new, model_path)
print(predictions)

[18.373768 24.050972 23.995586 18.601511 24.117937 17.8939   18.253143
 18.00199  18.002289 18.00199  22.762533 22.741997 22.741997 22.741997
 25.59785  25.59785  25.59785  25.59785  25.59785  25.59785 ]


In [13]:
df_predict = X_non_features[non_features].copy()
df_predict["Cab (µg/cm²)"] = predictions

In [14]:
df_predict

,timestamp,specimen,noisy_channels,Cab (µg/cm²)
0,2026-10-09T17:29:15,chilli_green_high,NaN,18.373768
1,2026-10-09T17:29:48,chilli_green_low,F1 (35.1%),24.050972
2,2026-10-09T17:30:08,chilli_green_low,F1 (52.7%),23.995586
3,2026-10-09T17:30:25,chilli_green_low,F1 (316.2%),18.601511
4,2026-10-09T17:30:45,chilli_green_low,F1 (105.4%); F3 (15.1%),24.117937
5,2026-10-12T11:21:56,cilli_old,F2 (52.7%); F3 (28.4%),17.893900
6,2026-10-12T11:22:15,cilli_old,F2 (86.1%),18.253143
7,2026-10-13T16:40:41,chilli_green_high,NaN,18.001989
8,2026-10-13T16:40:53,chilli_green_high,NaN,18.002289
9,2026-10-13T16:41:05,chilli_green_high,NaN,18.001989
